# Phishing Website Detection - URL-Enabled ML Project

This notebook introduces the raw-URL workflow. The full reproducible training/evaluation pipeline is in `src/train_model.py`; run that script from the project root to compare four models and generate report artifacts.

## 1. Project setup

Install dependencies from the project root:

```bash
python -m pip install -r requirements.txt
```

In [ ]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
print('Project root:', PROJECT_ROOT)

## 2. Load raw URL labels from UCI PhiUSIIL (ID 967)

The loader retains only the raw URL string and target label. The dataset's precomputed webpage fields are not used by this URL-only classifier.

In [ ]:
from src.data_loader import load_url_dataset

urls, y, provenance = load_url_dataset()
print(provenance)
print('URL records:', len(urls))
print('Class counts (0=legitimate, 1=phishing):')
print(y.value_counts().sort_index())

## 3. Convert a URL to numerical features

The extractor is shared by model training and the Streamlit application. It performs local string parsing only; it does not fetch the submitted website.

In [ ]:
from src.url_features import FEATURE_NAMES, extract_url_features, normalize_user_url

example_url = 'https://secure-login.example.com/account/verify?user=123&next=%2Fhome'
example_features = extract_url_features(normalize_user_url(example_url))
pd.Series(example_features, name='feature_value').to_frame()

## 4. Train and evaluate the four classifiers

From the terminal at the project root, run:

```bash
python src/train_model.py
```

The script uses a host-group-aware train/test split and 5-fold stratified group cross-validation. It compares Logistic Regression, Decision Tree, Linear SVM and Gaussian Naive Bayes, then saves the selected model and evaluation artifacts.

In [ ]:
comparison_path = PROJECT_ROOT / 'artifacts' / 'model_comparison.csv'
if comparison_path.exists():
    comparison = pd.read_csv(comparison_path)
    display(comparison)
else:
    print('Run python src/train_model.py first to generate model_comparison.csv')

## 5. Try the trained model on a new URL

The Streamlit app provides the same workflow with a URL text box:

```bash
streamlit run app.py
```

Optional webpage inspection is separate and user-triggered. Its HTML observations do not feed the URL classifier.

## 6. Generate the project report

After training, run:

```bash
python generate_report.py
```

The report uses the actual metrics saved by your local run. Do not invent or copy metrics from another dataset run.